# import 및 데이터 불러오기



In [2]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.9 MB/s eta 0:00:00


In [4]:
# CatBoost 패키지 설치
!pip install catboost -q
from scipy.optimize import minimize
from catboost import CatBoostClassifier
print("CatBoost 및 추가 모듈 설치/로드 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00
CatBoost 및 추가 모듈 설치/로드 완료!


In [5]:
import re
import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [6]:
train = pd.read_csv('train.csv', encoding = 'utf-8-sig')
test = pd.read_csv('test.csv', encoding = 'utf-8-sig')
print(f"데이터 로드 완료")

데이터 로드 완료


# 전처리 완료 데이터

In [7]:
X_train = train.drop(columns=["ID", "시술 시기 코드","임신 시도 또는 마지막 임신 경과 연수", "임신 성공 여부"])
y_train = train["임신 성공 여부"]
X_test = test.drop(columns = ['ID', '시술 시기 코드', '임신 시도 또는 마지막 임신 경과 연수']).copy()

In [8]:
# 결측치 처리 전 결측치 확인
missing = X_train.isnull().sum()
print(missing[missing > 0].to_string())

특정 시술 유형                 2
단일 배아 이식 여부           6291
착상 전 유전 검사 사용 여부    253633
착상 전 유전 진단 사용 여부      6291
배아 생성 주요 이유           6291
총 생성 배아 수             6291
미세주입된 난자 수            6291
미세주입에서 생성된 배아 수       6291
이식된 배아 수              6291
미세주입 배아 이식 수          6291
저장된 배아 수              6291
미세주입 후 저장된 배아 수       6291
해동된 배아 수              6291
해동 난자 수               6291
수집된 신선 난자 수           6291
저장된 신선 난자 수           6291
혼합된 난자 수              6291
파트너 정자와 혼합된 난자 수      6291
기증자 정자와 혼합된 난자 수      6291
동결 배아 사용 여부           6291
신선 배아 사용 여부           6291
기증 배아 사용 여부           6291
대리모 여부                6291
PGD 시술 여부           254172
PGS 시술 여부           254422
난자 채취 경과일            57488
난자 해동 경과일           254915
난자 혼합 경과일            53735
배아 이식 경과일            43566
배아 해동 경과일           215982


In [ ]:
# '특정 시술 유형' 결측치 처리
X_train["특정 시술 유형"] = X_train["특정 시술 유형"].fillna("Unknown")
X_test["특정 시술 유형"] = X_test["특정 시술 유형"].fillna("Unknown")
print(X_train["특정 시술 유형"].isnull().sum())
print(X_test["특정 시술 유형"].isnull().sum())

print(f"처리 결과: 결측치 처리 완료")

0
0
처리 결과: 결측치 처리 완료


In [ ]:
# 0,1 bool 범주형 컬럼 결측치 처리
pg_features = [
    "착상 전 유전 검사 사용 여부",
    "PGD 시술 여부",
    "PGS 시술 여부"
]

for col in pg_features:
    X_train[col] = X_train[col].fillna(0)
    X_test[col] = X_test[col].fillna(0)


pg_features = [
    "착상 전 유전 검사 사용 여부",
    "PGD 시술 여부",
    "PGS 시술 여부"
]

print(X_train[pg_features].apply(pd.Series.value_counts).fillna(0).astype(int))

print(f"처리 결과: 결측치 처리 완료")

     착상 전 유전 검사 사용 여부  PGD 시술 여부  PGS 시술 여부
0.0            253633     254172     254422
1.0              2718       2179       1929
처리 결과: 결측치 처리 완료


In [ ]:
# 0, nan으로 구성된 문자형 컬럼 결측치 처리
date_features = [
    "난자 채취 경과일",
    "난자 해동 경과일",
    "난자 혼합 경과일",
    "배아 이식 경과일",
    "배아 해동 경과일"
]

for col in date_features:
    X_train[col] = X_train[col].fillna(-1)
    X_test[col] = X_test[col].fillna(-1)

for col in date_features:
    print(f"{col}")
    print("train:", X_train[col].value_counts(dropna=False).sort_index().to_dict())
    print("test :", X_test[col].value_counts(dropna=False).sort_index().to_dict())

print(f"처리 결과: 결측치 처리 완료")

난자 채취 경과일
train: {-1.0: 57488, 0.0: 198863}
test : {-1.0: 19949, 0.0: 70118}
난자 해동 경과일
train: {-1.0: 254915, 0.0: 1434, 1.0: 2}
test : {-1.0: 89575, 0.0: 491, 5.0: 1}
난자 혼합 경과일
train: {-1.0: 53735, 0.0: 201920, 1.0: 488, 2.0: 102, 3.0: 68, 4.0: 5, 5.0: 24, 6.0: 8, 7.0: 1}
test : {-1.0: 18579, 0.0: 71213, 1.0: 201, 2.0: 38, 3.0: 23, 4.0: 5, 5.0: 5, 6.0: 2, 7.0: 1}
배아 이식 경과일
train: {-1.0: 43566, 0.0: 24904, 1.0: 6053, 2.0: 35078, 3.0: 57924, 4.0: 4504, 5.0: 81459, 6.0: 2773, 7.0: 90}
test : {-1.0: 15246, 0.0: 8538, 1.0: 2094, 2.0: 12206, 3.0: 20630, 4.0: 1570, 5.0: 28790, 6.0: 960, 7.0: 33}
배아 해동 경과일
train: {-1.0: 215982, 0.0: 39801, 1.0: 37, 2.0: 182, 3.0: 164, 4.0: 12, 5.0: 140, 6.0: 30, 7.0: 3}
test : {-1.0: 76117, 0.0: 13751, 1.0: 17, 2.0: 57, 3.0: 59, 4.0: 4, 5.0: 55, 6.0: 7}
처리 결과: 결측치 처리 완료


In [ ]:
# 6,291개의 결측치 행 처리하기 ()
# 같은 행인지(연관성이 있는지) 확인
missing_features = [
    "단일 배아 이식 여부",
    "착상 전 유전 진단 사용 여부",
    "배아 생성 주요 이유",
    "총 생성 배아 수",
    "미세주입된 난자 수",
    "미세주입에서 생성된 배아 수",
    "이식된 배아 수",
    "미세주입 배아 이식 수",
    "저장된 배아 수",
    "미세주입 후 저장된 배아 수",
    "해동된 배아 수",
    "해동 난자 수",
    "수집된 신선 난자 수",
    "저장된 신선 난자 수",
    "혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수",
    "기증자 정자와 혼합된 난자 수",
    "동결 배아 사용 여부",
    "신선 배아 사용 여부",
    "기증 배아 사용 여부",
    "대리모 여부"
]

print(X_train[missing_features].isnull().all(axis=1).sum())
print(X_test[missing_features].isnull().all(axis=1).sum())

print(f"결측치 확인 결과: 결측치가 모두 동일한 행에 존재합니다")

In [ ]:
# 숫자형 컬럼 처리
numeric_missing = [
    "총 생성 배아 수",
    "미세주입된 난자 수",
    "미세주입에서 생성된 배아 수",
    "이식된 배아 수",
    "미세주입 배아 이식 수",
    "저장된 배아 수",
    "미세주입 후 저장된 배아 수",
    "해동된 배아 수",
    "해동 난자 수",
    "수집된 신선 난자 수",
    "저장된 신선 난자 수",
    "혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수",
    "기증자 정자와 혼합된 난자 수"
]

for col in numeric_missing:
    X_train[col] = X_train[col].fillna(-1)
    X_test[col] = X_test[col].fillna(-1)

print(f"처리 결과: 결측치 처리 완료")

처리 결과: 결측치 처리 완료


In [ ]:
# 논리형 컬럼 처리
bool_missing = [
    "단일 배아 이식 여부",
    "착상 전 유전 진단 사용 여부",
    "동결 배아 사용 여부",
    "신선 배아 사용 여부",
    "기증 배아 사용 여부",
    "대리모 여부"
]

for col in bool_missing:
    X_train[col] = X_train[col].fillna(-1)
    X_test[col] = X_test[col].fillna(-1)

print(f"처리 결과: 결측치 처리 완료")

처리 결과: 결측치 처리 완료


In [ ]:
# 범주형 컬럼 처리
X_train["배아 생성 주요 이유"] = (
    X_train["배아 생성 주요 이유"].fillna("Unknown")
)

X_test["배아 생성 주요 이유"] = (
    X_test["배아 생성 주요 이유"].fillna("Unknown"))

print(f"처리 결과: 결측치 처리 완료")

처리 결과: 결측치 처리 완료


In [ ]:
# 횟수형으로 표시된 컬럼 숫자형으로 전환
count_columns = [
    "총 시술 횟수",
    "클리닉 내 총 시술 횟수",
    "IVF 시술 횟수",
    "DI 시술 횟수",
    "총 임신 횟수",
    "IVF 임신 횟수",
    "DI 임신 횟수",
    "총 출산 횟수",
    "IVF 출산 횟수",
    "DI 출산 횟수"
]

for col in count_columns:
    X_train[col] = (
        X_train[col]
        .str.replace("회", "", regex=False)
        .str.replace("이상", "", regex=False)
        .astype(int)
    )

    X_test[col] = (
        X_test[col]
        .str.replace("회", "", regex=False)
        .str.replace("이상", "", regex=False)
        .astype(int)
    )
print(f"횟수형(문자형) 컬럼 숫자형으로 전환 완료")
print(f"6회 이상 -> 6으로 바뀌어 약간의 정보 손실이 일어났습니다")

횟수형(문자형) 컬럼 숫자형으로 전환 완료
6회 이상 -> 6으로 바뀌어 약간의 정보 손실이 일어났습니다


In [ ]:
# 결측치 처리 최종 확인
print("train 결측치:")
print(X_train.isnull().sum().sort_values(ascending=False).head(20))

print("\ntest 결측치:")
print(X_test.isnull().sum().sort_values(ascending=False).head(20))

train 결측치:
시술 당시 나이            0
시술 유형               0
특정 시술 유형            0
배란 자극 여부            0
배란 유도 유형            0
단일 배아 이식 여부         0
착상 전 유전 검사 사용 여부    0
착상 전 유전 진단 사용 여부    0
남성 주 불임 원인          0
남성 부 불임 원인          0
여성 주 불임 원인          0
여성 부 불임 원인          0
부부 주 불임 원인          0
부부 부 불임 원인          0
불명확 불임 원인           0
불임 원인 - 난관 질환       0
불임 원인 - 남성 요인       0
불임 원인 - 배란 장애       0
불임 원인 - 여성 요인       0
불임 원인 - 자궁경부 문제     0
dtype: int64

test 결측치:
시술 당시 나이            0
시술 유형               0
특정 시술 유형            0
배란 자극 여부            0
배란 유도 유형            0
단일 배아 이식 여부         0
착상 전 유전 검사 사용 여부    0
착상 전 유전 진단 사용 여부    0
남성 주 불임 원인          0
남성 부 불임 원인          0
여성 주 불임 원인          0
여성 부 불임 원인          0
부부 주 불임 원인          0
부부 부 불임 원인          0
불명확 불임 원인           0
불임 원인 - 난관 질환       0
불임 원인 - 남성 요인       0
불임 원인 - 배란 장애       0
불임 원인 - 여성 요인       0
불임 원인 - 자궁경부 문제     0
dtype: int64


In [ ]:
categorical_columns = [
    "시술 유형", "특정 시술 유형", "배란 유도 유형", "배아 생성 주요 이유",
    "난자 출처", "정자 출처", "난자 기증자 나이", "정자 기증자 나이", "시술 당시 나이"
]

numeric_columns = [
    "총 시술 횟수", "클리닉 내 총 시술 횟수", "IVF 시술 횟수", "DI 시술 횟수",
    "총 임신 횟수", "IVF 임신 횟수", "DI 임신 횟수",
    "총 출산 횟수", "IVF 출산 횟수", "DI 출산 횟수",

    # 배아/난자 관련 수량
    "총 생성 배아 수", "미세주입된 난자 수", "미세주입에서 생성된 배아 수",
    "이식된 배아 수", "미세주입 배아 이식 수", "저장된 배아 수",
    "미세주입 후 저장된 배아 수", "해동된 배아 수",
    "해동 난자 수", "수집된 신선 난자 수", "저장된 신선 난자 수",
    "혼합된 난자 수", "파트너 정자와 혼합된 난자 수", "기증자 정자와 혼합된 난자 수",

    # 경과일
    "난자 채취 경과일", "난자 해동 경과일", "난자 혼합 경과일",
    "배아 이식 경과일", "배아 해동 경과일"
]

bool_features = [
    "배란 자극 여부", "단일 배아 이식 여부", "착상 전 유전 검사 사용 여부",
    "착상 전 유전 진단 사용 여부", "남성 주 불임 원인", "남성 부 불임 원인",
    "여성 주 불임 원인", "여성 부 불임 원인", "부부 주 불임 원인",
    "부부 부 불임 원인", "불명확 불임 원인",
    "불임 원인 - 난관 질환", "불임 원인 - 남성 요인", "불임 원인 - 배란 장애",
    "불임 원인 - 여성 요인", "불임 원인 - 자궁경부 문제", "불임 원인 - 자궁내막증",
    "불임 원인 - 정자 농도", "불임 원인 - 정자 면역학적 요인", "불임 원인 - 정자 운동성",
    "불임 원인 - 정자 형태", "동결 배아 사용 여부", "신선 배아 사용 여부",
    "기증 배아 사용 여부", "대리모 여부", "PGD 시술 여부", "PGS 시술 여부",
    "관련 정보 결측 여부"
]

In [ ]:
# 카테고리형 컬럼들을 문자열로 변환
for col in categorical_columns:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(X_train[categorical_columns].dtypes)
print(f"카테고리형 컬럼들 문자열 변환 확인 완료")

시술 유형          object
특정 시술 유형       object
배란 유도 유형       object
배아 생성 주요 이유    object
난자 출처          object
정자 출처          object
난자 기증자 나이      object
정자 기증자 나이      object
시술 당시 나이       object
dtype: object
카테고리형 컬럼들 문자열 변환 확인 완료


In [ ]:
# OneHotEncoder를 사용하여 카테코리형 변수 인코딩
# 범주 간 순서 관계를 만들지 않아 one-hot을 베이스로 진행
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
X_train_encoded = encoder.fit_transform(X_train[categorical_columns])
X_test_encdoed = encoder.transform(X_test[categorical_columns])

encoded_columns = encoder.get_feature_names_out(categorical_columns)
X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoded_columns)
X_test_encoded = pd.DataFrame(X_test_encdoed, columns=encoded_columns)

X_train_final = pd.concat([X_train.drop(columns=categorical_columns), X_train_encoded], axis=1)
X_test_final = pd.concat([X_test.drop(columns=categorical_columns), X_test_encoded], axis=1)

print(f"처리 결과: One-Hot 인코딩 완료")

처리 결과: One-Hot 인코딩 완료


In [ ]:
# 컬럼명 오류 발생 후 수정
import re

def clean_feature_names(columns):
    return [
        re.sub(r'[^A-Za-z0-9가-힣_]', '_', str(col))
        for col in columns
    ]

X_train_final.columns = clean_feature_names(
    X_train_final.columns
)

X_test_final.columns = X_train_final.columns

In [ ]:
# 데이터 처리 최종 확인
print("X_train_final shape:", X_train_final.shape)
print("X_test_final shape :", X_test_final.shape)

print("\nData types:")
print(X_train_final.dtypes.value_counts())

print("\nMissing values:")
print("Train:", X_train_final.isna().sum().sum())
print("Test :", X_test_final.isna().sum().sum())

print("\ny_train shape:", y_train.shape)
print("\ny_train distribution:")
print(y_train.value_counts())
print("\ny_train ratio:")
print(y_train.value_counts(normalize=True))

print(f"처리 결과: ✅ 머신 러닝 용 데이터 처리 완료")

X_train_final shape: (256351, 126)
X_test_final shape : (90067, 126)

Data types:
float64    98
int64      28
Name: count, dtype: int64

Missing values:
Train: 0
Test : 0

y_train shape: (256351,)

y_train distribution:
임신 성공 여부
0    190123
1     66228
Name: count, dtype: int64

y_train ratio:
임신 성공 여부
0    0.741651
1    0.258349
Name: proportion, dtype: float64
처리 결과: ✅ 머신 러닝 용 데이터 처리 완료


# 데이터 분할

In [ ]:
# 데이터 분리
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Train:", X_tr.shape)
print("Validation:", X_val.shape)

print("\nTrain Target 분포:")
print(y_tr.value_counts())

print("\nValidation Target 분포:")
print(y_val.value_counts())

Train: (205080, 126)
Validation: (51271, 126)

Train Target 분포:
임신 성공 여부
0    152098
1     52982
Name: count, dtype: int64

Validation Target 분포:
임신 성공 여부
0    38025
1    13246
Name: count, dtype: int64


In [ ]:
# 분리된 데이터 중복 확인
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(X_tr.shape, X_val.shape)
print("중복:", X_tr.columns.duplicated().sum())

(205080, 126) (51271, 126)
중복: 0


# 기본 단일 모델 학습

In [ ]:
# XGboost 학습
xgb_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.07,
    max_depth=3,
    min_child_weight=2,
    subsample=0.9,
    colsample_bytree=1.0,
    gamma=0.4,
    reg_alpha=1.5,
    reg_lambda=2,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

print("XGBoost 학습 시작...")

xgb_model.fit(
    X_tr,
    y_tr
)

print("XGBoost 학습 완료!")

xgb_prob = xgb_model.predict_proba(X_val)[:, 1]

xgb_roc_auc = roc_auc_score(y_val, xgb_prob)
xgb_pr_auc = average_precision_score(y_val, xgb_prob)

print(f"XGBoost ROC-AUC : {xgb_roc_auc:.4f}")
print(f"XGBoost PR-AUC  : {xgb_pr_auc:.4f}")

XGBoost 학습 시작...
XGBoost 학습 완료!
XGBoost ROC-AUC : 0.7359
XGBoost PR-AUC  : 0.4474


In [ ]:
# LightGBM 학습
lgb_model = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

print("LightGBM 학습 시작...")

lgb_model.fit(
    X_tr,
    y_tr
)

print("LightGBM 학습 완료!")

lgb_prob = lgb_model.predict_proba(X_val)[:, 1]

lgb_roc_auc = roc_auc_score(y_val, lgb_prob)
lgb_pr_auc = average_precision_score(y_val, lgb_prob)

print(f"LightGBM ROC-AUC : {lgb_roc_auc:.4f}")
print(f"LightGBM PR-AUC  : {lgb_pr_auc:.4f}")

LightGBM 학습 시작...
LightGBM 학습 완료!
LightGBM ROC-AUC : 0.7354
LightGBM PR-AUC  : 0.4448


In [ ]:
# CatBoost 학습
cat_model = CatBoostClassifier(
    iterations=400,
    learning_rate=0.07,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100,
    thread_count=-1
)

print("CatBoost 학습 시작...")

cat_model.fit(
    X_tr,
    y_tr,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50
)

print("CatBoost 학습 완료!")

cat_prob = cat_model.predict_proba(X_val)[:, 1]

cat_roc_auc = roc_auc_score(y_val, cat_prob)
cat_pr_auc = average_precision_score(y_val, cat_prob)

print(f"CatBoost ROC-AUC : {cat_roc_auc:.4f}")
print(f"CatBoost PR-AUC  : {cat_pr_auc:.4f}")

CatBoost 학습 시작...
0:	test: 0.7070801	best: 0.7070801 (0)	total: 113ms	remaining: 45.2s
100:	test: 0.7345007	best: 0.7345007 (100)	total: 7.22s	remaining: 21.4s
200:	test: 0.7357925	best: 0.7358097 (198)	total: 20.6s	remaining: 20.4s
300:	test: 0.7358469	best: 0.7358925 (289)	total: 27.6s	remaining: 9.08s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7358924883
bestIteration = 289

Shrink model to first 290 iterations.
CatBoost 학습 완료!
CatBoost ROC-AUC : 0.7359
CatBoost PR-AUC  : 0.4477


In [ ]:
# 3개 모델 비교
model_results = pd.DataFrame({
    "Model": [
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],
    "ROC-AUC": [
        xgb_roc_auc,
        lgb_roc_auc,
        cat_roc_auc
    ],
    "PR-AUC": [
        xgb_pr_auc,
        lgb_pr_auc,
        cat_pr_auc
    ]
})

model_results = model_results.sort_values(
    "ROC-AUC",
    ascending=False
).reset_index(drop=True)

model_results.insert(
    0,
    "Rank",
    range(1, len(model_results) + 1)
)

display(model_results)

,Rank,Model,ROC-AUC,PR-AUC
0,1,XGBoost,0.735897,0.447371
1,2,CatBoost,0.735892,0.447665
2,3,LightGBM,0.735376,0.444808


# 하이퍼 파라미터 최적화 (optuna로 처리)

In [ ]:
# 공통 설정
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
# ---------- XGBoost (GPU) ----------
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "eval_metric": "auc",
        "tree_method": "hist",
        "device": "cuda",  # XGBoost 2.x GPU 지정 방식 (구버전이면 tree_method="gpu_hist" 사용)
    }

    scores = []
    for tr_idx, val_idx in skf.split(X_train_final, y_train):
        X_tr, X_val = X_train_final.iloc[tr_idx], X_train_final.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, pred))

    return np.mean(scores)


study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=50)

print("[XGBoost] Best ROC-AUC:", study_xgb.best_value)
print("[XGBoost] Best Params:", study_xgb.best_params)

In [ ]:
# ---------- LightGBM (GPU) ----------
def objective_lgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "device": "gpu",       # GPU 빌드가 아니면 여기서 에러 발생
        "gpu_platform_id": 0,
        "gpu_device_id": 0,
    }

    scores = []
    for tr_idx, val_idx in skf.split(X_train_final, y_train):
        X_tr, X_val = X_train_final.iloc[tr_idx], X_train_final.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, pred))

    return np.mean(scores)


study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=50)

print("[LightGBM] Best ROC-AUC:", study_lgb.best_value)
print("[LightGBM] Best Params:", study_lgb.best_params)

In [ ]:
# ---------- CatBoost (GPU) ----------
def objective_cat(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 5.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": 42,
        "task_type": "GPU",     # GPU가 아니면 task_type="CPU"로 변경
        "devices": "0",
        "verbose": False,
    }

    scores = []
    for tr_idx, val_idx in skf.split(X_train_final, y_train):
        X_tr, X_val = X_train_final.iloc[tr_idx], X_train_final.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, pred))

    return np.mean(scores)


study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(objective_cat, n_trials=50)

print("[CatBoost] Best ROC-AUC:", study_cat.best_value)
print("[CatBoost] Best Params:", study_cat.best_params)

In [ ]:
# ===== HPO에서 찾은 최적 파라미터 =====

xgb_params = {
    'n_estimators': 498,
    'learning_rate': 0.04056465457643643,
    'max_depth': 6,
    'min_child_weight': 5,
    'subsample': 0.7365601875280953,
    'colsample_bytree': 0.6258218790630266,
    'gamma': 3.4778492643213017,
    'reg_alpha': 3.023983772556902,
    'reg_lambda': 2.1072264050611587
}

lgb_params = {
    'n_estimators': 517,
    'learning_rate': 0.023519899815177964,
    'num_leaves': 20,
    'max_depth': 9,
    'min_child_samples': 99,
    'subsample': 0.9831855237044063,
    'colsample_bytree': 0.6936293442632957,
    'reg_alpha': 3.7325104289033835,
    'reg_lambda': 2.7090617717415286
}

cat_params = {
    'iterations': 675,
    'learning_rate': 0.03679309163277342,
    'depth': 6,
    'l2_leaf_reg': 8.668598434408018,
    'bagging_temperature': 0.39604424997419874,
    'random_strength': 1.9379972336837308,
    'border_count': 226
}

In [ ]:
# LogisticRegression 최적화된 파라미터 값
lr_params = {
    'max_iter': 2500,
    'solver': 'lbfgs',
    'C': 1.0,
    'random_state': 42
}

In [ ]:
# 최적화된 파라미터 값 추가

# 앙상블 XGB + LGBM + Cat + Logistic
최적화된 파라미터로 모델 학습

In [ ]:
# XGBoost 최적 모델
xgb_best = XGBClassifier(
    **xgb_params,
    random_state=42,
    eval_metric="auc",
    tree_method="hist",
    device="cuda"
)

# LightGBM 최적 모델
lgb_best = LGBMClassifier(
    **lgb_params,
    random_state=42,
    device="gpu",
    gpu_platform_id=0,
    gpu_device_id=0
)

# CatBoost 최적 모델
cat_best = CatBoostClassifier(
    **cat_params,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    task_type="GPU",
    devices="0",
    verbose=False
)

print("XGBoost 학습 시작")
xgb_best.fit(X_train_final, y_train)
print("XGBoost 학습 완료")

print("\nLightGBM 학습 시작")
lgb_best.fit(X_train_final, y_train)
print("LightGBM 학습 완료")

print("\nCatBoost 학습 시작")
cat_best.fit(X_train_final, y_train)
print("CatBoost 학습 완료")

XGBoost 학습 시작
XGBoost 학습 완료

LightGBM 학습 시작
LightGBM 학습 완료

CatBoost 학습 시작


Default metric period is 5 because AUC is/are not implemented for GPU


CatBoost 학습 완료


In [ ]:
# Logistic Regression 학습
# 데이터 스케일링 적용 (X_train_final 스케일링)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

# 학습 실행
print("Logistic Regression 학습 시작")
lr_best = LogisticRegression(**lr_params)
lr_best.fit(X_train_scaled, y_train)
print("Logistic Regression 학습 완료")

Logistic Regression 학습 시작
Logistic Regression 학습 완료


In [ ]:
# 새로운 모델 학습 추가

# OOF 생성

In [ ]:
# XGBoost + LightGBM + CatBoost OOF 생성
# ==========================================

def generate_oof_tree_models(X, y, skf):

    n_train = len(X)

    oof_xgb = np.zeros(n_train)
    oof_lgb = np.zeros(n_train)
    oof_cat = np.zeros(n_train)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):

        print(f"\n========== Fold {fold+1}/{skf.get_n_splits()} ==========")

        X_tr = X.iloc[tr_idx]
        X_val = X.iloc[val_idx]

        y_tr = y.iloc[tr_idx]
        y_val = y.iloc[val_idx]

        # ------------------------------
        # XGBoost
        # ------------------------------
        xgb_model = XGBClassifier(
            **xgb_params,
            random_state=42,
            eval_metric="auc",
            tree_method="hist",
            device="cuda"
        )

        xgb_model.fit(X_tr, y_tr)

        xgb_pred = xgb_model.predict_proba(X_val)[:, 1]
        oof_xgb[val_idx] = xgb_pred

        print(
            f"XGBoost ROC-AUC: "
            f"{roc_auc_score(y_val, xgb_pred):.4f}"
        )

        # ------------------------------
        # LightGBM
        # ------------------------------
        lgb_model = LGBMClassifier(
            **lgb_params,
            random_state=42,
            device="gpu",
            gpu_platform_id=0,
            gpu_device_id=0,
            verbosity=-1
        )

        lgb_model.fit(X_tr, y_tr)

        lgb_pred = lgb_model.predict_proba(X_val)[:, 1]
        oof_lgb[val_idx] = lgb_pred

        print(
            f"LightGBM ROC-AUC: "
            f"{roc_auc_score(y_val, lgb_pred):.4f}"
        )

        # ------------------------------
        # CatBoost
        # ------------------------------
        cat_model = CatBoostClassifier(
            **cat_params,
            random_seed=42,
            verbose=False,
            task_type="GPU"
        )

        cat_model.fit(X_tr, y_tr)

        cat_pred = cat_model.predict_proba(X_val)[:, 1]
        oof_cat[val_idx] = cat_pred

        print(
            f"CatBoost ROC-AUC: "
            f"{roc_auc_score(y_val, cat_pred):.4f}"
        )

    return oof_xgb, oof_lgb, oof_cat


# OOF 생성
print("XGBoost + LightGBM + CatBoost OOF 생성 시작")

oof_xgb, oof_lgb, oof_cat = generate_oof_tree_models(
    X_train_final,
    y_train,
    skf
)

# 저장
np.save("oof_xgb.npy", oof_xgb)
np.save("oof_lgb.npy", oof_lgb)
np.save("oof_cat.npy", oof_cat)

print("\nOOF 저장 완료")

XGBoost + LightGBM + CatBoost OOF 생성 시작

========== Fold 1/5 ==========
XGBoost ROC-AUC: 0.7368
LightGBM ROC-AUC: 0.7372
CatBoost ROC-AUC: 0.7370

========== Fold 2/5 ==========
XGBoost ROC-AUC: 0.7420
LightGBM ROC-AUC: 0.7420
CatBoost ROC-AUC: 0.7420

========== Fold 3/5 ==========
XGBoost ROC-AUC: 0.7395
LightGBM ROC-AUC: 0.7396
CatBoost ROC-AUC: 0.7389

========== Fold 4/5 ==========
XGBoost ROC-AUC: 0.7375
LightGBM ROC-AUC: 0.7376
CatBoost ROC-AUC: 0.7375

========== Fold 5/5 ==========
XGBoost ROC-AUC: 0.7402
LightGBM ROC-AUC: 0.7402
CatBoost ROC-AUC: 0.7398

OOF 저장 완료


In [ ]:
# Logistic Regression OOF 생성 (fold별 스케일링 포함)

def generate_oof_lr(model_class, params, X_train, y_train, skf, model_name="Logistic Regression"):
    n_train = X_train.shape[0]
    oof_pred = np.zeros(n_train)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        fold_scaler = StandardScaler()
        X_tr_scaled = fold_scaler.fit_transform(X_tr)
        X_val_scaled = fold_scaler.transform(X_val)

        model = model_class(**params)
        model.fit(X_tr_scaled, y_tr)

        val_pred = model.predict_proba(X_val_scaled)[:, 1]
        oof_pred[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        print(f"[{model_name}] Fold {fold+1}/{skf.get_n_splits()} ROC-AUC: {fold_auc:.4f}")

    overall_auc = roc_auc_score(y_train, oof_pred)
    print(f"[{model_name}] 전체 OOF ROC-AUC: {overall_auc:.4f}\n")
    return oof_pred


print("Logistic Regression OOF 생성 시작")

oof_lr = generate_oof_lr(
    LogisticRegression,
    lr_params,
    X_train_final,
    y_train,
    skf,
    model_name="Logistic Regression"
)

# OOF 저장
np.save("oof_lr.npy", oof_lr)

print("Logistic Regression OOF 저장 완료")

Logistic Regression OOF 생성 시작
[Logistic Regression] Fold 1/5 ROC-AUC: 0.7219
[Logistic Regression] Fold 2/5 ROC-AUC: 0.7272
[Logistic Regression] Fold 3/5 ROC-AUC: 0.7233
[Logistic Regression] Fold 4/5 ROC-AUC: 0.7213
[Logistic Regression] Fold 5/5 ROC-AUC: 0.7238
[Logistic Regression] 전체 OOF ROC-AUC: 0.7235

Logistic Regression OOF 저장 완료


In [ ]:
print("\n=== OOF 값 확인 ===")

print(
    "XGB:",
    oof_xgb.min(),
    oof_xgb.max(),
    oof_xgb.mean()
)

print(
    "LGB:",
    oof_lgb.min(),
    oof_lgb.max(),
    oof_lgb.mean()
)

print(
    "Cat:",
    oof_cat.min(),
    oof_cat.max(),
    oof_cat.mean()
)

print("\n=== 전체 OOF ROC-AUC ===")

print(
    "XGB:",
    roc_auc_score(y_train, oof_xgb)
)

print(
    "LGB:",
    roc_auc_score(y_train, oof_lgb)
)

print(
    "Cat:",
    roc_auc_score(y_train, oof_cat)
)


=== OOF 값 확인 ===
XGB: 0.0005085923476144671 0.74899822473526 0.2583179190158106
LGB: 0.0004609293365809976 0.7277103017668639 0.2583299947277655
Cat: 0.00020673796012252776 0.7762140382385972 0.2582636006748611

=== 전체 OOF ROC-AUC ===
XGB: 0.7392080199775664
LGB: 0.7392858233879538
Cat: 0.7390416931183521


In [ ]:
# 새로운 모델 들어올 시 OOF 따로 추가

# test 데이터 예측


In [ ]:
# test 데이터 예측
xgb_prob = xgb_best.predict_proba(X_test_final)[:, 1]
lgb_prob = lgb_best.predict_proba(X_test_final)[:, 1]
cat_prob = cat_best.predict_proba(X_test_final)[:, 1]

print("XGBoost 예측 완료")
print("LightGBM 예측 완료")
print("CatBoost 예측 완료")

print("\nXGBoost 확률 범위:",
      xgb_prob.min(), "~", xgb_prob.max())

print("LightGBM 확률 범위:",
      lgb_prob.min(), "~", lgb_prob.max())

print("CatBoost 확률 범위:",
      cat_prob.min(), "~", cat_prob.max())

XGBoost 예측 완료
LightGBM 예측 완료
CatBoost 예측 완료

XGBoost 확률 범위: 0.0004819051 ~ 0.7469788
LightGBM 확률 범위: 0.0005037938957376324 ~ 0.707180141014266
CatBoost 확률 범위: 0.00022598673567807438 ~ 0.7424803512999981


In [ ]:
lr_prob = lr_best.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression 예측 완료")

print("\nLogistic Regression 확률 범위:",
      lr_prob.min(), "~", lr_prob.max())

Logistic Regression 예측 완료

Logistic Regression 확률 범위: 4.7661487268091035e-08 ~ 0.8699101882525982


# 만들어진 OOF로 가중치 계산하기

In [ ]:
# 이전 단계에서 저장한 OOF 예측 불러오기 (또는 이미 변수로 갖고 있다면 그대로 사용)

# 4개 모델 가중치 조합 생성 (합이 1이 되도록, step = 0.1)
# ==========================================

step = 0.1

weights = []

for xgb_w in np.arange(0.0, 1.0 + step, step):
    for lgb_w in np.arange(0.0, 1.0 - xgb_w + step, step):
        for cat_w in np.arange(
            0.0,
            1.0 - xgb_w - lgb_w + step,
            step
        ):
            lr_w = round(1.0 - xgb_w - lgb_w - cat_w, 2)

            if lr_w < 0:
                continue

            weights.append((
                round(xgb_w, 2),
                round(lgb_w, 2),
                round(cat_w, 2),
                lr_w
            ))


print(f"총 가중치 조합 수: {len(weights)}")


# ==========================================
# 가중치별 OOF 성능 비교
# ==========================================

print("\n=== 4개 모델 가중치별 OOF 성능 비교 ===")

results = []

for xgb_w, lgb_w, cat_w, lr_w in weights:

    blended_oof = (
        xgb_w * oof_xgb +
        lgb_w * oof_lgb +
        cat_w * oof_cat +
        lr_w * oof_lr
    )

    auc = roc_auc_score(y_train, blended_oof)
    pr_auc = average_precision_score(y_train, blended_oof)

    results.append((
        xgb_w,
        lgb_w,
        cat_w,
        lr_w,
        auc,
        pr_auc
    ))


# ==========================================
# ROC-AUC 기준 상위 결과 확인
# ==========================================

results_sorted = sorted(
    results,
    key=lambda x: x[4],
    reverse=True
)

print("\n=== ROC-AUC 기준 상위 10개 ===")

for result in results_sorted[:10]:

    xgb_w, lgb_w, cat_w, lr_w, auc, pr_auc = result

    print(
        f"XGB {xgb_w:.1f} : "
        f"LGB {lgb_w:.1f} : "
        f"Cat {cat_w:.1f} : "
        f"LR {lr_w:.1f} "
        f"→ ROC-AUC {auc:.4f} | PR-AUC {pr_auc:.4f}"
    )


# ==========================================
# 단일 모델 baseline
# ==========================================

auc_xgb_only = roc_auc_score(y_train, oof_xgb)
auc_lgb_only = roc_auc_score(y_train, oof_lgb)
auc_cat_only = roc_auc_score(y_train, oof_cat)
auc_lr_only = roc_auc_score(y_train, oof_lr)

print("\n=== 단일 모델 OOF ROC-AUC ===")
print(f"XGBoost             : {auc_xgb_only:.4f}")
print(f"LightGBM            : {auc_lgb_only:.4f}")
print(f"CatBoost            : {auc_cat_only:.4f}")
print(f"Logistic Regression : {auc_lr_only:.4f}")


# ==========================================
# 최적 가중치 선택
# ROC-AUC 기준
# ==========================================

(
    best_xgb_w,
    best_lgb_w,
    best_cat_w,
    best_lr_w,
    best_auc,
    best_pr_auc
) = max(results, key=lambda x: x[4])


print("\n=== 최적 가중치 ===")

print(
    f"XGBoost {best_xgb_w:.1f} : "
    f"LightGBM {best_lgb_w:.1f} : "
    f"CatBoost {best_cat_w:.1f} : "
    f"Logistic Regression {best_lr_w:.1f}"
)

print(f"ROC-AUC : {best_auc:.4f}")
print(f"PR-AUC  : {best_pr_auc:.4f}")


# ==========================================
# 최적 가중치로 최종 Test 예측
# 전체 데이터로 학습한 모델의 test 확률 사용
# ==========================================

final_blended_prob = (
    best_xgb_w * xgb_prob +
    best_lgb_w * lgb_prob +
    best_cat_w * cat_prob +
    best_lr_w * lr_prob
)


print(
    "\n최종 앙상블 확률 범위:",
    final_blended_prob.min(),
    "~",
    final_blended_prob.max()
)

총 가중치 조합 수: 286

=== 4개 모델 가중치별 OOF 성능 비교 ===

=== ROC-AUC 기준 상위 10개 ===
XGB 0.2 : LGB 0.5 : Cat 0.3 : LR 0.0 → ROC-AUC 0.7394 | PR-AUC 0.4500
XGB 0.3 : LGB 0.4 : Cat 0.3 : LR -0.0 → ROC-AUC 0.7394 | PR-AUC 0.4500
XGB 0.1 : LGB 0.6 : Cat 0.3 : LR -0.0 → ROC-AUC 0.7394 | PR-AUC 0.4500
XGB 0.2 : LGB 0.4 : Cat 0.4 : LR 0.0 → ROC-AUC 0.7394 | PR-AUC 0.4500
XGB 0.3 : LGB 0.5 : Cat 0.2 : LR -0.0 → ROC-AUC 0.7394 | PR-AUC 0.4500
XGB 0.1 : LGB 0.5 : Cat 0.4 : LR 0.0 → ROC-AUC 0.7394 | PR-AUC 0.4500
XGB 0.2 : LGB 0.6 : Cat 0.2 : LR -0.0 → ROC-AUC 0.7394 | PR-AUC 0.4500
XGB 0.4 : LGB 0.3 : Cat 0.3 : LR -0.0 → ROC-AUC 0.7394 | PR-AUC 0.4501
XGB 0.4 : LGB 0.4 : Cat 0.2 : LR -0.0 → ROC-AUC 0.7394 | PR-AUC 0.4500
XGB 0.3 : LGB 0.3 : Cat 0.4 : LR -0.0 → ROC-AUC 0.7394 | PR-AUC 0.4501

=== 단일 모델 OOF ROC-AUC ===
XGBoost             : 0.7392
LightGBM            : 0.7393
CatBoost            : 0.7390
Logistic Regression : 0.7235

=== 최적 가중치 ===
XGBoost 0.2 : LightGBM 0.5 : CatBoost 0.3 : Logistic Regressi

In [ ]:
# 가중치 비율 설정
ensemble_prob = (
    best_xgb_w * xgb_prob +
    best_lgb_w * lgb_prob +
    best_cat_w * cat_prob +
    best_lr_w * lr_prob
)

print("앙상블 예측 완료")

print("\n최종 가중치")
print(f"XGBoost : {best_xgb_w:.1f}")
print(f"LightGBM : {best_lgb_w:.1f}")
print(f"CatBoost : {best_cat_w:.1f}")
print(f"Logistic Regression : {best_lr_w:.1f}")

print("\n평균 확률:", ensemble_prob.mean())

print(
    "확률 범위:",
    ensemble_prob.min(),
    "~",
    ensemble_prob.max())

앙상블 예측 완료

최종 가중치
XGBoost : 0.2
LightGBM : 0.5
CatBoost : 0.3
Logistic Regression : 0.0

평균 확률: 0.2583493410678306
확률 범위: 0.0004545019943306112 ~ 0.7131712034805896


# 모델 성능 비교


In [ ]:
# 최종 모델 성능 비교표
# ROC-AUC / PR-AUC / Recall / Precision / F1

# 1. 최종 앙상블 OOF 생성

final_oof = (
    0.2 * oof_xgb +
    0.5 * oof_lgb +
    0.3 * oof_cat +
    0.0 * oof_lr
)

# 0.5 기준 분류
final_pred = (final_oof >= 0.5).astype(int)


# 2. 모델별 OOF 예측값

model_oof = {
    "XGBoost": oof_xgb,
    "LightGBM": oof_lgb,
    "CatBoost": oof_cat,
    "Logistic Regression": oof_lr,
    "최종 앙상블": final_oof
}

# 3. 성능 계산

results = []

for model_name, pred_prob in model_oof.items():

    pred = (pred_prob >= 0.5).astype(int)

    results.append({
        "모델": model_name,
        "ROC-AUC": roc_auc_score(y_train, pred_prob),
        "PR-AUC": average_precision_score(y_train, pred_prob),
        "Recall": recall_score(
            y_train,
            pred,
            zero_division=0
        ),
        "Precision": precision_score(
            y_train,
            pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_train,
            pred,
            zero_division=0
        )
    })


# 4. DataFrame 생성

final_results_df = pd.DataFrame(results)

metric_cols = [
    "ROC-AUC",
    "PR-AUC",
    "Recall",
    "Precision",
    "F1"
]

final_results_df[metric_cols] = (
    final_results_df[metric_cols]
    .round(4)
)

# 5. 출력

print("=== 최종 모델 성능 비교 ===")
display(final_results_df)

=== 최종 모델 성능 비교 ===


,모델,ROC-AUC,PR-AUC,Recall,Precision,F1
0,XGBoost,0.7392,0.4497,0.1111,0.5431,0.1845
1,LightGBM,0.7393,0.4496,0.1031,0.5461,0.1735
2,CatBoost,0.7390,0.4494,0.1124,0.5430,0.1863
3,Logistic Regression,0.7235,0.4233,0.0987,0.4984,0.1648
4,최종 앙상블,0.7394,0.4500,0.1066,0.5454,0.1783


# cvs 파일로 저장

In [ ]:
# 파일 제출
sample_submission = pd.read_csv('./sample_submission.csv')

sample_submission["probability"] = ensemble_prob

sample_submission.to_csv(
    'final_ensemble_xgb_lgb_cat.csv',
    index=False,
    encoding='utf-8-sig'
)

print("제출 파일 저장 완료!")
print("파일명: final_ensemble_xgb_lgb_cat.csv")
print("행 수:", len(sample_submission))
print(sample_submission.head())

제출 파일 저장 완료!
파일명: final_ensemble_xgb_lgb_cat.csv
행 수: 90067
           ID  probability
0  TEST_00000     0.001854
1  TEST_00001     0.004025
2  TEST_00002     0.161228
3  TEST_00003     0.110715
4  TEST_00004     0.503874


# 시각화 자료들 모음

In [ ]:
# '임신 시도 또는 마지막 임신 경과 연수'를 제외 한 이유 1 (결측치가 너무 많다)
import matplotlib.pyplot as plt

column = '임신 시도 또는 마지막 임신 경과 연수'

missing_count = df[column].isna().sum()
non_missing_count = df[column].notna().sum()

values = [missing_count, non_missing_count]
labels = ['결측', '비결측']

plt.figure(figsize=(7, 7))

def make_label(label, count):
    return f'{label}\n(n={count:,})'

pie_labels = [
    make_label('결측', missing_count),
    make_label('비결측', non_missing_count)
]

plt.pie(
    values,
    labels=pie_labels,
    autopct='%1.2f%%',
    startangle=90,
    colors=['#E57373', '#64B5F6'],
    textprops={'fontsize': 11}
)

plt.title(
    "'임신 시도 또는 마지막 임신 경과 연수' 결측치 현황",
    fontsize=14
)

plt.show()

In [ ]:
# '임신 시도 또는 마지막 임신 경과 연수'를 제외 한 이유 2 (결측치를 정의하기가 어렵다)
# '임신 시도 또는 마지막 임신 경과 연수'가 결측인 행만 추출
missing_group = df[
    df['임신 시도 또는 마지막 임신 경과 연수'].isna()
].copy()

# 전체 결측 그룹 수
total = len(missing_group)

# 데이터 전처리: '회', ' 이상' 제거 및 숫자형 변환
for col in ['총 임신 횟수', '총 시술 횟수', '총 출산 횟수']:
    if missing_group[col].dtype == 'object':
        missing_group[col] = missing_group[col].astype(str).str.replace('회', '').str.replace(' 이상', '')
        missing_group[col] = pd.to_numeric(missing_group[col], errors='coerce')

# 1회 이상 경험한 사람의 비율 계산
result = {
    '임신 1회 이상 경험': (
        (missing_group['총 임신 횟수'] >= 1).sum() / total * 100
    ),
    '시술 1회 이상 경험': (
        (missing_group['총 시술 횟수'] >= 1).sum() / total * 100
    ),
    '출산 1회 이상 경험': (
        (missing_group['총 출산 횟수'] >= 1).sum() / total * 100
    )
}

# 결과 확인
print(f"결측 그룹: {total:,}건\n")

for key, value in result.items():
    print(f"{key}: {value:.2f}%")

# -----------------------------
# 시각화
# -----------------------------

labels = list(result.keys())
values = list(result.values())

plt.figure(figsize=(8, 5))

bars = plt.bar(
    labels,
    values,
    color=['#6BAED6', '#74C476', '#FD8D3C'],
    width=0.6
)

plt.ylabel('비율 (%)')
plt.title(
    "'임신 시도 또는 마지막 임신 경과 연수' 결측 그룹의 경험 여부"
)

plt.ylim(0, 100)

# 막대 위에 비율 표시
for bar, value in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f'{value:.2f}%',
        ha='center',
        va='bottom',
        fontsize=11
    )

plt.tight_layout()
plt.show()

In [ ]:
# 기증자 나이에서 결측치 '알 수 없음'은 임의 처리 없이 따로 문자형으로 구분
# 기증자 나이 알 수 없음에서 난자 정자 본인 제공시 비율이 높기 때문에 임의 처리하지 않는 것이 낫다고 판단됨
# 난자: 기증자 나이가 '알 수 없음'인 데이터 중 '본인 제공' 비율
egg_unknown = df[df['난자 기증자 나이'] == '알 수 없음']

egg_self_rate = (
    (egg_unknown['난자 출처'] == '본인 제공').mean() * 100
)

# 정자: 기증자 나이가 '알 수 없음'인 데이터 중 '배우자 제공' 비율
sperm_unknown = df[df['정자 기증자 나이'] == '알 수 없음']

sperm_partner_rate = (
    (sperm_unknown['정자 출처'] == '배우자 제공').mean() * 100
)

print(f"난자 기증자 나이 '알 수 없음' 중 본인 제공: {egg_self_rate:.2f}%")
print(f"정자 기증자 나이 '알 수 없음' 중 배우자 제공: {sperm_partner_rate:.2f}%")

In [ ]:
# 기본 세팅으로 단독 모델링 accuracy 값 비교

models = ['XGBoost', 'LightGBM', 'Logistic Reg.', 'CatBoost', 'ExtraTrees', 'Random Forest', 'Random Forest', 'Ensemble']
accuracy = [0.7453, 0.7450, 0.7408, 0.7361, 0.7214, 0.7157, 0.6866, 0.7381]

plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracy, color='skyblue')
plt.ylim(0.70, 0.755)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison (Vertical)')
plt.xticks(rotation=30)

for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
             f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 횟수 관련 컬럼들 히트맵
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

# '횟수'가 들어간 컬럼 선택
count_cols = [col for col in train.columns if '횟수' in col]

count_data = train[count_cols].copy()

# 문자열을 숫자로 변환
def convert_count(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    # '6 이상' → 6
    if '이상' in x:
        return float(re.findall(r'\d+', x)[0])

    # '0회', '1회' → 숫자만 추출
    numbers = re.findall(r'\d+\.?\d*', x)

    if numbers:
        return float(numbers[0])

    return np.nan

for col in count_data.columns:
    count_data[col] = count_data[col].apply(convert_count)

# 상관관계
corr = count_data.corr()

# 위쪽 삼각형 제거 → 왼쪽 아래만 표시
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 10))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True
)

plt.title("횟수 관련 변수 상관관계 히트맵")
plt.tight_layout()
plt.show()